# 06 — Statistical Analysis

Research question: **which rhythm features differ significantly between the
eight folk genres?**

This notebook performs non-parametric omnibus and post-hoc statistical
testing on the segment-level rhythm feature dataframe produced in
`02_Rhythm_Feature_Extraction.ipynb`. It is a statistical comparison of
feature distributions, **not** a classification experiment, and statistical
significance reported here does not by itself establish musical or
cultural significance, nor causation.

In [1]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from itertools import combinations

from scipy import stats
from statsmodels.stats.multitest import multipletests

sys.path.append("..")
from utils.similarity import get_feature_columns

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
ALPHA = 0.05


In [2]:
DATA_PATH = Path("../data/extracted/rhythm_features.pkl")
TABLES_DIR = Path("../tables")
FIGURES_DIR = Path("../figures")
TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


## 1. Load rhythm feature dataframe

In [3]:
with open(DATA_PATH, "rb") as f:
    import pickle
    rhythm_df = pickle.load(f)

rhythm_df.shape


(109089, 41)

## 2. Identify numerical rhythm feature columns

In [4]:
METADATA_COLS = ["genre", "state", "artist", "gender", "song", "source", "source_file"]
metadata_cols = [c for c in METADATA_COLS if c in rhythm_df.columns]
feature_cols = get_feature_columns(rhythm_df, exclude=metadata_cols)

GENRES = ["Bauls", "Bhatiali", "Gidha", "Kajri", "Maand", "Sohar", "Sufi", "Uttarakhandi"]
present_genres = [g for g in GENRES if g in rhythm_df["genre"].unique()]

len(feature_cols), present_genres


(34,
 ['Bauls',
  'Bhatiali',
  'Gidha',
  'Kajri',
  'Maand',
  'Sohar',
  'Sufi',
  'Uttarakhandi'])

In [5]:
analysis_df = rhythm_df[rhythm_df["genre"].isin(present_genres)].reset_index(drop=True)
analysis_df = analysis_df.dropna(subset=feature_cols)
analysis_df = analysis_df[np.isfinite(analysis_df[feature_cols].to_numpy()).all(axis=1)]
len(rhythm_df), len(analysis_df)


(109089, 109089)

**Note on unit of observation:** `analysis_df` contains one row per
*segment*, and a single song may contribute multiple segments. Treating
every segment as a fully independent observation understates variance and
can inflate apparent significance (pseudoreplication). The segment-level
tests below (Sections 5-9) are reported as the primary, full-power analysis,
but Section 12 repeats the key test at the **song level** (one row per
song, using `song` and `genre` metadata) as a more conservative,
non-pseudoreplicated robustness check.

## 3. Descriptive statistics by genre

In [6]:
def iqr(x: pd.Series) -> float:
    return float(np.percentile(x, 75) - np.percentile(x, 25))

desc_records = []
for feature in feature_cols:
    grouped = analysis_df.groupby("genre")[feature]
    for genre, values in grouped:
        desc_records.append({
            "feature": feature,
            "genre": genre,
            "n": int(values.shape[0]),
            "mean": float(values.mean()),
            "median": float(values.median()),
            "std": float(values.std()),
            "iqr": iqr(values),
        })

descriptive_stats_df = pd.DataFrame(desc_records)
descriptive_stats_df.to_csv(TABLES_DIR / "descriptive_statistics_by_genre.csv", index=False)
descriptive_stats_df.head(10)


,feature,genre,n,mean,median,std,iqr
0,energy_mean,Bauls,8450,-5572.577910,-5337.471680,868.228088,1012.607666
1,energy_mean,Bhatiali,8808,-5513.188648,-5453.315918,879.725343,1338.873291
2,energy_mean,Gidha,8228,-5399.633279,-5460.587891,598.247830,898.126587
3,energy_mean,Kajri,20580,-5340.463634,-5188.163086,833.282637,1103.370361
4,energy_mean,Maand,17796,-4958.984325,-4869.266113,620.948419,708.040894
5,energy_mean,Sohar,12994,-5549.617932,-5422.554932,649.041258,876.429077
6,energy_mean,Sufi,19213,-5038.084448,-4939.562500,716.430985,816.189941
7,energy_mean,Uttarakhandi,13020,-4762.562376,-4748.630371,508.639127,618.403687
8,energy_std,Bauls,8450,528.765570,492.193481,240.721912,190.456993
9,energy_std,Bhatiali,8808,490.255562,450.605759,242.889303,198.514793


## 4. Assumption checks

In [7]:
SHAPIRO_SUBSAMPLE_N = 500

normality_records = []
rng = np.random.default_rng(RANDOM_STATE)

for feature in feature_cols:
    for genre in present_genres:
        values = analysis_df.loc[analysis_df["genre"] == genre, feature].to_numpy()
        if len(values) > SHAPIRO_SUBSAMPLE_N:
            idx = rng.choice(len(values), size=SHAPIRO_SUBSAMPLE_N, replace=False)
            values = values[idx]
        stat, p_value = stats.shapiro(values)
        normality_records.append({"feature": feature, "genre": genre, "shapiro_p": p_value})

normality_df = pd.DataFrame(normality_records)
pct_non_normal = float((normality_df["shapiro_p"] < ALPHA).mean() * 100)
normality_df.to_csv(TABLES_DIR / "normality_check_shapiro.csv", index=False)
pct_non_normal


C:\Users\ankit\miniconda3\envs\ml\lib\site-packages\scipy\stats\_axis_nan_policy.py:586: UserWarning: scipy.stats.shapiro: Input data has range zero. The results may not be accurate.
  res = hypotest_fun_out(*samples, **kwds)


93.01470588235294

In [8]:
levene_records = []
for feature in feature_cols:
    groups = [analysis_df.loc[analysis_df["genre"] == g, feature].to_numpy() for g in present_genres]
    stat, p_value = stats.levene(*groups)
    levene_records.append({"feature": feature, "levene_stat": stat, "levene_p": p_value})

levene_df = pd.DataFrame(levene_records)
pct_unequal_variance = float((levene_df["levene_p"] < ALPHA).mean() * 100)
levene_df.to_csv(TABLES_DIR / "variance_homogeneity_levene.csv", index=False)
pct_unequal_variance


C:\Users\ankit\miniconda3\envs\ml\lib\site-packages\scipy\stats\_morestats.py:3057: RuntimeWarning: invalid value encountered in scalar divide
  W = numer / denom


97.05882352941177

Based on the subsampled Shapiro-Wilk checks and full-sample Levene's
tests above, a substantial proportion of feature-genre distributions
deviate from normality and/or show unequal variances across genres
(percentages computed in the two cells above). This supports the use of
**Kruskal-Wallis** (rank-based, does not assume normality or equal
variances) as the primary omnibus test, as specified for this analysis.

## 5–6. Kruskal-Wallis omnibus test per feature

In [9]:
#import scipy.stats as stats

kw_records = []
skipped_features = []

for feature in feature_cols:
    groups = [
        analysis_df.loc[analysis_df["genre"] == g, feature]
        .dropna()
        .to_numpy()
        for g in present_genres
    ]

    # Remove empty groups
    groups = [g for g in groups if len(g) > 0]

    # Check whether the feature has any variation
    all_values = np.concatenate(groups)

    if len(all_values) == 0 or np.unique(all_values).size < 2:
        skipped_features.append(feature)
        continue

    try:
        h_stat, p_value = stats.kruskal(*groups)

        n_total = sum(len(g) for g in groups)
        k_groups = len(groups)

        eta_sq_h = max(
            0.0,
            (h_stat - k_groups + 1) / (n_total - k_groups)
        )

        kw_records.append({
            "feature": feature,
            "H_statistic": h_stat,
            "p_value": p_value,
            "n_total": n_total,
            "k_groups": k_groups,
            "eta_squared_H": eta_sq_h,
        })

    except ValueError as e:
        skipped_features.append(feature)

kw_df = pd.DataFrame(kw_records)

# Apply formatting and drop the k_groups column
if not kw_df.empty:
    kw_df['p_value'] = kw_df['p_value'].apply(lambda x: '< 0.001' if x < 0.001 else f"{x:.3f}")
    kw_df = kw_df.drop(columns=['k_groups'])

print(f"Tested features: {len(kw_df)}")
print(f"Skipped features: {len(skipped_features)}")

if skipped_features:
    print("\nSkipped constant features:")
    for feature in skipped_features:
        print(" -", feature)

kw_df.head()

Tested features: 33
Skipped features: 1

Skipped constant features:
 - tempogram_stability


,feature,H_statistic,p_value,n_total,eta_squared_H
0,energy_mean,15721.350285,< 0.001,109089,0.144061
1,energy_std,5408.366022,< 0.001,109089,0.049517
2,energy_max,15038.415062,< 0.001,109089,0.137800
3,energy_min,16946.368384,< 0.001,109089,0.155292
4,energy_range,6379.908080,< 0.001,109089,0.058424


In [10]:
print(skipped_features)

['tempogram_stability']


In [11]:
print(kw_df.shape)

(33, 5)


## 7. Multiple-comparison correction (Benjamini-Hochberg / FDR)

In [ ]:
reject, p_adjusted, _, _ = multipletests(kw_df["p_value"].to_numpy(), alpha=ALPHA, method="fdr_bh")
kw_df["p_value_fdr_bh"] = p_adjusted
kw_df["significant_after_fdr"] = reject

kw_df.to_csv(TABLES_DIR / "kruskal_wallis_omnibus_results.csv", index=False)
kw_df.sort_values("p_value_fdr_bh").head(10)


## 8. Statistically significant features after correction

In [ ]:
significant_features_df = kw_df[kw_df["significant_after_fdr"]].copy()
significant_features_df = significant_features_df.sort_values(
    ["p_value_fdr_bh", "eta_squared_H"], ascending=[True, False]
).reset_index(drop=True)

significant_features_df.to_csv(TABLES_DIR / "significant_features_after_fdr.csv", index=False)
len(significant_features_df), significant_features_df.head(10)


## 9. Post-hoc pairwise comparisons (Mann-Whitney U) for significant features

In [ ]:
def rank_biserial_effect_size(x: np.ndarray, y: np.ndarray, u_stat: float) -> float:
    n1, n2 = len(x), len(y)
    return float(1 - (2 * u_stat) / (n1 * n2))

posthoc_records = []
genre_pairs = list(combinations(present_genres, 2))

for feature in significant_features_df["feature"]:
    pair_pvalues = []
    pair_meta = []
    for genre_a, genre_b in genre_pairs:
        x = analysis_df.loc[analysis_df["genre"] == genre_a, feature].to_numpy()
        y = analysis_df.loc[analysis_df["genre"] == genre_b, feature].to_numpy()
        u_stat, p_value = stats.mannwhitneyu(x, y, alternative="two-sided")
        effect = rank_biserial_effect_size(x, y, u_stat)
        pair_pvalues.append(p_value)
        pair_meta.append((genre_a, genre_b, u_stat, effect))

    pair_reject, pair_p_fdr, _, _ = multipletests(pair_pvalues, alpha=ALPHA, method="fdr_bh")

    for (genre_a, genre_b, u_stat, effect), p_fdr, sig in zip(pair_meta, pair_p_fdr, pair_reject):
        posthoc_records.append({
            "feature": feature,
            "genre_a": genre_a,
            "genre_b": genre_b,
            "U_statistic": u_stat,
            "p_value_fdr_bh": p_fdr,
            "rank_biserial_effect_size": effect,
            "significant_after_fdr": bool(sig),
        })

posthoc_df = pd.DataFrame(posthoc_records)
posthoc_df.to_csv(TABLES_DIR / "posthoc_pairwise_mannwhitney.csv", index=False)
posthoc_df.shape


Post-hoc p-values are FDR-corrected **within each feature** across its
28 genre pairs (Benjamini-Hochberg), separately from the omnibus-level
correction in Section 7. This two-stage correction is a common and
defensible approach but should be reported explicitly, as done here.

## 10–11. Ranking features by corrected significance and effect size

In [ ]:
ranked_features_df = significant_features_df.copy()
ranked_features_df["rank_by_significance"] = ranked_features_df["p_value_fdr_bh"].rank(method="min")
ranked_features_df["rank_by_effect_size"] = ranked_features_df["eta_squared_H"].rank(method="min", ascending=False)
ranked_features_df["combined_rank"] = (
    ranked_features_df["rank_by_significance"] + ranked_features_df["rank_by_effect_size"]
)
ranked_features_df = ranked_features_df.sort_values("combined_rank").reset_index(drop=True)

ranked_features_df.to_csv(TABLES_DIR / "ranked_significant_features.csv", index=False)
ranked_features_df.head(15)


## 12. Visualizations

In [ ]:
TOP_N = 10
top_features = ranked_features_df["feature"].head(TOP_N).tolist()

plt.figure(figsize=(9, 6))
plot_data = ranked_features_df.head(TOP_N).iloc[::-1]
plt.barh(plot_data["feature"], -np.log10(plot_data["p_value_fdr_bh"]), color="steelblue")
plt.xlabel("-log10(FDR-corrected p-value)")
plt.title("Top Significant Rhythm Features (Kruskal-Wallis, FDR-corrected)")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "top_significant_features_pvalue.png", dpi=300)
plt.show()


In [ ]:
plt.figure(figsize=(9, 6))
plt.barh(plot_data["feature"], plot_data["eta_squared_H"], color="darkorange")
plt.xlabel("Effect Size (eta-squared H)")
plt.title("Top Significant Rhythm Features Ranked by Effect Size")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "top_significant_features_effect_size.png", dpi=300)
plt.show()


In [ ]:
n_plot_features = min(6, len(top_features))
fig, axes = plt.subplots(2, 3, figsize=(17, 9))
axes = axes.flatten()

for ax, feature in zip(axes, top_features[:n_plot_features]):
    sns.violinplot(data=analysis_df, x="genre", y=feature, ax=ax, inner="box", cut=0)
    ax.set_title(feature)
    ax.tick_params(axis="x", rotation=45)

for ax in axes[n_plot_features:]:
    ax.axis("off")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "top_features_violin_by_genre.png", dpi=300)
plt.show()


In [ ]:
genre_means = analysis_df.groupby("genre")[top_features].mean()
genre_means_z = (genre_means - genre_means.mean(axis=0)) / genre_means.std(axis=0)

plt.figure(figsize=(11, 6))
sns.heatmap(genre_means_z.T, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Standardized Genre Means — Top Significant Rhythm Features")
plt.xlabel("Genre")
plt.ylabel("Feature")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "genre_means_heatmap_top_features.png", dpi=300)
plt.show()


In [ ]:
significance_summary = pd.DataFrame({
    "total_features_tested": [len(kw_df)],
    "significant_after_fdr": [int(kw_df["significant_after_fdr"].sum())],
    "pct_significant": [round(float(kw_df["significant_after_fdr"].mean() * 100), 2)],
})
significance_summary.to_csv(TABLES_DIR / "significance_summary.csv", index=False)
significance_summary


In [ ]:
plt.figure(figsize=(6, 6))
counts = kw_df["significant_after_fdr"].value_counts().rename({True: "Significant", False: "Not significant"})
plt.pie(counts, labels=counts.index, autopct="%1.1f%%", colors=["indianred", "lightgrey"])
plt.title("Proportion of Rhythm Features Significant After FDR Correction")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "significance_summary_pie.png", dpi=300)
plt.show()


## 13. Regional / state-level analysis (conditional)

In [ ]:
genre_state_crosstab = pd.crosstab(analysis_df["genre"], analysis_df["state"])
states_per_genre = (genre_state_crosstab > 0).sum(axis=1)
genres_per_state = (genre_state_crosstab > 0).sum(axis=0)

genre_state_crosstab.to_csv(TABLES_DIR / "genre_state_crosstab.csv")
states_per_genre, genres_per_state


In [ ]:
state_analysis_meaningful = bool((genres_per_state > 1).sum() >= 2 and (states_per_genre > 1).any())

if not state_analysis_meaningful:
    print(
        "State-level testing skipped: in this dataset, genre and state are "
        "effectively confounded (each state maps to essentially one genre, "
        "or vice versa, per the crosstab above). A state-level Kruskal-Wallis "
        "test would not be separable from the genre-level test already "
        "performed above, so it would be misleading to report it as an "
        "independent regional effect."
    )
else:
    state_kw_records = []
    present_states = analysis_df["state"].dropna().unique().tolist()
    for feature in feature_cols:
        groups = [analysis_df.loc[analysis_df["state"] == s, feature].to_numpy() for s in present_states]
        groups = [g for g in groups if len(g) > 0]
        if len(groups) < 2:
            continue
        h_stat, p_value = stats.kruskal(*groups)
        state_kw_records.append({"feature": feature, "H_statistic": h_stat, "p_value": p_value})

    state_kw_df = pd.DataFrame(state_kw_records)
    _, state_p_fdr, _, state_reject = multipletests(state_kw_df["p_value"], alpha=ALPHA, method="fdr_bh")
    state_kw_df["p_value_fdr_bh"] = state_p_fdr
    state_kw_df["significant_after_fdr"] = state_reject
    state_kw_df.to_csv(TABLES_DIR / "kruskal_wallis_state_level.csv", index=False)
    print(state_kw_df.sort_values("p_value_fdr_bh").head(10))


## 14. Song-level robustness check

In [ ]:
has_song_metadata = "song" in analysis_df.columns and analysis_df["song"].notna().any()
has_song_metadata


In [ ]:
if has_song_metadata:
    song_level_df = (
        analysis_df.groupby(["genre", "song"])[feature_cols]
        .mean()
        .reset_index()
    )

    song_kw_records = []
    for feature in ranked_features_df["feature"]:
        groups = [
            song_level_df.loc[song_level_df["genre"] == g, feature].to_numpy()
            for g in present_genres
        ]
        groups = [g for g in groups if len(g) > 0]
        if len(groups) < 2:
            continue
        h_stat, p_value = stats.kruskal(*groups)
        song_kw_records.append({"feature": feature, "H_statistic": h_stat, "p_value": p_value, "n_songs": len(song_level_df)})

    song_kw_df = pd.DataFrame(song_kw_records)
    if len(song_kw_df) > 0:
        _, song_p_fdr, _, song_reject = multipletests(song_kw_df["p_value"], alpha=ALPHA, method="fdr_bh")
        song_kw_df["p_value_fdr_bh"] = song_p_fdr
        song_kw_df["significant_after_fdr"] = song_reject

    song_kw_df.to_csv(TABLES_DIR / "kruskal_wallis_song_level_robustness.csv", index=False)
    song_kw_df.sort_values("p_value_fdr_bh")
else:
    print("Song metadata not usable for aggregation; song-level robustness check skipped.")


**Interpretation of the song-level check:** this repeats the
Kruskal-Wallis omnibus test on **one row per song** (segment-level features
averaged within each song), removing the pseudoreplication caused by
multiple segments per song. Features that remain significant after FDR
correction at *both* the segment level and the song level are the most
defensible candidates for genuinely differing rhythm characteristics
between genres. Features significant only at the segment level should be
interpreted cautiously, since their apparent significance may be partly
driven by the larger, non-independent segment-level sample size.

## Summary and scientific caveats

- Kruskal-Wallis was used as the primary omnibus test because rhythm
  features are not reliably normally distributed and do not reliably have
  homogeneous variances across genres (Section 4).
- P-values were corrected for multiple comparisons using Benjamini-Hochberg
  FDR at the omnibus level (Section 7) and, separately, within each
  significant feature's pairwise post-hoc comparisons (Section 9).
- Effect sizes (eta-squared H for the omnibus test, rank-biserial
  correlation for pairwise comparisons) are reported alongside p-values,
  since statistical significance alone does not indicate the size or
  practical importance of a difference.
- A state-level test was only run if genre and state were not effectively
  confounded in this dataset (Section 13); otherwise it was skipped and the
  reason was reported.
- A song-level robustness check (Section 14) addresses the fact that
  segments from the same song are not independent observations.
- None of the above results should be interpreted as evidence of
  causation, as a classification result, or as a musicological or cultural
  conclusion beyond what the numerical feature comparisons directly show.